# IS 507 資料前處理與完整度分析

本 notebook 使用組員共享的兩個單一 Parquet 檔：

- `data/asi_opensource_pod_hourly_day0_29.parquet`
- `data/asi_opensource_job_execution_summary.parquet`

整體流程分成三個部分：

1. 檢查兩份資料的檔案完整度、缺值與例外值。
2. 建立不改動原始檔的前處理規則，並預留 `workload_id` 的保留、補 0、刪列與序數化選項。
3. 產生欄位、數值分布與類別分布的 data profile 表格。

所有清理都透過 DuckDB views 在查詢時執行，不會覆寫原始 Parquet，也不會另外複製一份 30 多 GiB 的 processed dataset。


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "MID" / "data").exists():
    ROOT = ROOT / "MID"

DATA_DIR = ROOT / "data"
POD_PARQUET = DATA_DIR / "asi_opensource_pod_hourly_day0_29.parquet"
JOB_PARQUET = DATA_DIR / "asi_opensource_job_execution_summary.parquet"

assert POD_PARQUET.exists(), f"Missing pod-hourly file: {POD_PARQUET}"
assert JOB_PARQUET.exists(), f"Missing execution-summary file: {JOB_PARQUET}"

con = duckdb.connect()
con.execute("SET threads=4")
con.execute("SET memory_limit='8GB'")
TEMP_DIR = DATA_DIR / ".duckdb_tmp"
TEMP_DIR.mkdir(parents=True, exist_ok=True)
con.execute(f"SET temp_directory='{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order=true")

con.execute(f'''CREATE OR REPLACE VIEW pod_raw AS
                SELECT * FROM read_parquet('{POD_PARQUET.as_posix()}')''')
con.execute(f'''CREATE OR REPLACE VIEW job_raw AS
                SELECT * FROM read_parquet('{JOB_PARQUET.as_posix()}')''')

# Quick diagnostic scope. Expand these values only for a deliberate full profile.
POD_PROFILE_DAYS = [0]
POD_PROFILE_HOURS = [0]
JOB_PROFILE_ROW_LIMIT = 1_000_000

profile_days_sql = ", ".join(map(str, POD_PROFILE_DAYS))
profile_hours_sql = ", ".join(map(str, POD_PROFILE_HOURS))
con.execute(f'''CREATE OR REPLACE VIEW pod_profile_raw AS
                SELECT * FROM pod_raw
                WHERE day IN ({profile_days_sql})
                  AND hour IN ({profile_hours_sql})''')
con.execute(f'''CREATE OR REPLACE VIEW job_profile_raw AS
                SELECT * FROM job_raw LIMIT {JOB_PROFILE_ROW_LIMIT}''')

print("Pod-hourly:", POD_PARQUET)
print("Execution summary:", JOB_PARQUET)
print("Quick pod profile:", POD_PROFILE_DAYS, POD_PROFILE_HOURS)
print("Quick summary rows:", f"{JOB_PROFILE_ROW_LIMIT:,}")


## 1. 資料完整度、缺值與例外值

檔案 metadata、總列數，以及 pod 的 `day` / `hour` coverage 會檢查完整資料。為了讓 notebook 能在一般筆電上快速執行，缺值與分布 profile 預設使用 pod Day 0 / Hour 0，以及 execution summary 的前 1,000,000 列。

因此，完整度結果代表完整資料；quick profile 的百分比只代表清楚標示的檢查範圍，不應直接當成 30 天母體比例。


In [ ]:
EXPECTED_POD_ROWS = 842_390_418
EXPECTED_JOB_ROWS = 40_522_321

def parquet_metadata(path):
    parquet = pq.ParquetFile(path)
    return {
        "file": path.name,
        "size_gib": path.stat().st_size / 1024**3,
        "rows": parquet.metadata.num_rows,
        "row_groups": parquet.metadata.num_row_groups,
        "columns": parquet.metadata.num_columns,
    }

file_inventory = pd.DataFrame([
    parquet_metadata(POD_PARQUET),
    parquet_metadata(JOB_PARQUET),
])
display(file_inventory)

assert int(file_inventory.loc[0, "rows"]) == EXPECTED_POD_ROWS
assert int(file_inventory.loc[1, "rows"]) == EXPECTED_JOB_ROWS

pod_coverage = con.execute('''
SELECT
    count(*) AS rows,
    min(day) AS min_day,
    max(day) AS max_day,
    count(DISTINCT day) AS distinct_days,
    min(hour) AS min_hour,
    max(hour) AS max_hour,
    count(DISTINCT (day, hour)) AS distinct_day_hours
FROM pod_raw
''').df()
display(pod_coverage)

coverage = pod_coverage.iloc[0]
assert int(coverage["rows"]) == EXPECTED_POD_ROWS
assert (int(coverage["min_day"]), int(coverage["max_day"])) == (0, 29)
assert int(coverage["distinct_days"]) == 30
assert (int(coverage["min_hour"]), int(coverage["max_hour"])) == (0, 23)
assert int(coverage["distinct_day_hours"]) == 720
print("Completeness checks passed.")


In [ ]:
pod_quality = con.execute('''
SELECT
    count(*) AS row_count,
    count(DISTINCT pod_id) AS unique_pods,
    count(*) FILTER (WHERE workload_id IS NULL) AS missing_workload_id,
    count(*) FILTER (WHERE workload_id = '') AS blank_workload_id,
    count(*) FILTER (WHERE avg_gpu_sm_util IS NULL) AS missing_avg_gpu_sm_util,
    count(*) FILTER (WHERE schedule_delay_sec IS NULL) AS missing_schedule_delay,
    count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec IS NULL) AS missing_ready_delay,
    count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay
FROM pod_profile_raw
''').df()

job_quality = con.execute('''
SELECT
    count(*) AS row_count,
    count(DISTINCT pod_id) AS unique_pods,
    count(*) FILTER (WHERE workload_id IS NULL) AS missing_workload_id,
    count(*) FILTER (WHERE workload_id = '') AS blank_workload_id,
    count(*) FILTER (WHERE duration_hours IS NULL) AS missing_duration,
    count(*) FILTER (WHERE duration_hours < 0) AS negative_duration,
    count(*) FILTER (WHERE schedule_delay_sec IS NULL) AS missing_schedule_delay,
    count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec IS NULL) AS missing_ready_delay,
    count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay
FROM job_profile_raw
''').df()

def quality_table(frame, dataset):
    row = frame.iloc[0]
    total = int(row["row_count"])
    return pd.DataFrame([
        {
            "dataset": dataset,
            "metric": metric,
            "count": int(value),
            "percent_of_rows": 100.0 * float(value) / total,
        }
        for metric, value in row.items()
    ])

pod_scope_label = f"pod_hourly (days {POD_PROFILE_DAYS}, hours {POD_PROFILE_HOURS})"
job_scope_label = f"execution_summary (first {JOB_PROFILE_ROW_LIMIT:,} rows)"
quality_report = pd.concat([
    quality_table(pod_quality, pod_scope_label),
    quality_table(job_quality, job_scope_label),
], ignore_index=True)
display(quality_report)


### 判讀原則

- 不因任一欄位有缺值就刪除整列。
- 字串 `Unknown` / `unknown` 是來源資料的有效類別，不視為 `NULL`。
- 原始 delay 欄位保留；負值只會在新增的 `_clean` 欄位中轉為缺值。
- GPU utilization 的缺值率應搭配 `gpu_request > 0` 再判讀，因為大量 pod-hour rows 沒有申請 GPU。
- 極端 duration、utilization 或 GPU request 先保留並描述，不自動截尾。


## 2. 查詢時前處理與 `workload_id` 預留方案

本步驟只建立 views，不會改動或複製大型原始檔。所有原始欄位都會保留，清理後欄位使用新的名稱。

`WORKLOAD_ID_POLICY` 可設定為：

- `keep`：保留缺值，作為目前預設。
- `zero`：分析欄位中的缺值以字串 `"0"` 表示，同時保留缺值指標。
- `drop`：只在分析 view 中排除缺少 workload ID 的 rows，不刪除原始資料。
- `ordinal`：先建立小型 lookup table，再將已知 workload ID 編為 1、2、3……；缺值保留為 0。

序數只是匿名分析代碼，沒有大小或順序意義。建立完整 lookup 需要掃描兩份資料，因此函式已預留但不會自動執行。


In [ ]:
con.execute('''
CREATE OR REPLACE VIEW pod_analysis AS
SELECT
    *,
    nullif(trim(workload_id), '') AS workload_id_clean,
    CASE
        WHEN workload_id IS NULL OR trim(workload_id) = '' THEN 'missing'
        ELSE 'observed'
    END AS workload_id_status,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END
        AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END
        AS ready_delay_sec_clean,
    gpu_request > 0 AS is_gpu_request
FROM pod_raw
''')

con.execute('''
CREATE OR REPLACE VIEW job_analysis AS
SELECT
    *,
    nullif(trim(workload_id), '') AS workload_id_clean,
    CASE
        WHEN workload_id IS NULL OR trim(workload_id) = '' THEN 'missing'
        ELSE 'observed'
    END AS workload_id_status,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END
        AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END
        AS ready_delay_sec_clean
FROM job_raw
''')

con.execute('''
CREATE OR REPLACE VIEW pod_profile_analysis AS
SELECT * FROM pod_analysis
WHERE day IN (0) AND hour IN (0)
''')

con.execute('''
CREATE OR REPLACE VIEW job_profile_analysis AS
SELECT * FROM job_analysis LIMIT 1000000
''')

WORKLOAD_ID_POLICY = "keep"  # keep, zero, drop, or ordinal
WORKLOAD_LOOKUP_FILE = DATA_DIR / "workload_id_lookup.parquet"

def build_workload_id_lookup(output_path=WORKLOAD_LOOKUP_FILE):
    # Build a small shared ordinal lookup only if the team selects ordinal.
    con.execute(f'''
        COPY (
            WITH ids AS (
                SELECT workload_id_clean
                FROM pod_analysis
                WHERE workload_id_clean IS NOT NULL
                UNION
                SELECT workload_id_clean
                FROM job_analysis
                WHERE workload_id_clean IS NOT NULL
            )
            SELECT
                workload_id_clean,
                row_number() OVER (ORDER BY workload_id_clean) AS workload_id_ordinal
            FROM ids
        ) TO '{output_path.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    ''')
    return output_path

def create_preprocessed_views(policy=WORKLOAD_ID_POLICY):
    # Create lightweight views without rewriting either large Parquet file.
    valid = {"keep", "zero", "drop", "ordinal"}
    if policy not in valid:
        raise ValueError(f"policy must be one of {sorted(valid)}")

    if policy == "keep":
        expression = "workload_id_clean"
        where_clause = ""
        join_clause = ""
    elif policy == "zero":
        expression = "coalesce(workload_id_clean, '0')"
        where_clause = ""
        join_clause = ""
    elif policy == "drop":
        expression = "workload_id_clean"
        where_clause = "WHERE workload_id_clean IS NOT NULL"
        join_clause = ""
    else:
        if not WORKLOAD_LOOKUP_FILE.exists():
            raise FileNotFoundError(
                "Run build_workload_id_lookup() before selecting ordinal."
            )
        expression = "coalesce(lookup.workload_id_ordinal, 0)"
        where_clause = ""
        join_clause = (
            f"LEFT JOIN read_parquet('{WORKLOAD_LOOKUP_FILE.as_posix()}') AS lookup "
            "USING (workload_id_clean)"
        )

    for source, target in [
        ("pod_analysis", "pod_preprocessed"),
        ("job_analysis", "job_preprocessed"),
    ]:
        con.execute(f'''
            CREATE OR REPLACE VIEW {target} AS
            SELECT
                source.*, 
                source.workload_id_clean IS NULL AS workload_id_was_missing,
                {expression} AS workload_id_analysis
            FROM {source} AS source
            {join_clause}
            {where_clause}
        ''')

    return policy

active_workload_id_policy = create_preprocessed_views()

preprocessing_policy = pd.DataFrame([
    ["workload_id", f"Active policy: {active_workload_id_policy}"],
    ["schedule_delay_sec", "Keep raw; add non-negative *_clean field"],
    ["ready_delay_sec", "Keep raw; add non-negative *_clean field"],
    ["Unknown categories", "Keep as documented categories"],
    ["numeric outliers", "Keep and profile before any research-specific rule"],
])
preprocessing_policy.columns = ["field", "current_rule"]
display(preprocessing_policy)


In [ ]:
workload_id_coverage = con.execute('''
SELECT
    'pod quick profile' AS dataset,
    count(*) AS row_count,
    count(*) FILTER (WHERE workload_id_clean IS NOT NULL) AS rows_with_workload_id,
    count(DISTINCT workload_id_clean) AS unique_workloads
FROM pod_profile_analysis
UNION ALL
SELECT
    'summary quick profile',
    count(*),
    count(*) FILTER (WHERE workload_id_clean IS NOT NULL),
    count(DISTINCT workload_id_clean)
FROM job_profile_analysis
''').df()
workload_id_coverage["coverage_percent"] = (
    100 * workload_id_coverage["rows_with_workload_id"]
    / workload_id_coverage["row_count"]
)
display(workload_id_coverage)


### `workload_id` 的目前決策

目前使用 `keep`：單表分析保留缺值，不做推測性補值。若研究問題需要 workload-level aggregation 或 join，應使用 `workload_id_clean` 的非缺值部分並報告保留比例。

不要直接用 `pod_id` 取代缺失的 workload ID，因為這會改變分析層級。明天若團隊決定其他策略，只需修改 `WORKLOAD_ID_POLICY`；原始 Parquet 不會受到影響。


## 3. 欄位與資料描述表

欄位 profile 會整理資料型態、非缺值數、缺值比例與範圍；numeric profile 另外提供平均數、標準差與精確分位數；categorical profile 則呈現前幾大類別與比例。

這些表格使用 setup cell 中明確標示的 quick-profile 範圍，並在最後輸出成小型 CSV 檔供組員查看。


In [ ]:
pod_column_profile = con.execute(
    "SUMMARIZE SELECT * FROM pod_profile_analysis"
).df()
pod_column_profile.insert(0, "dataset", pod_scope_label)

job_column_profile = con.execute(
    "SUMMARIZE SELECT * FROM job_profile_analysis"
).df()
job_column_profile.insert(0, "dataset", job_scope_label)

column_profile = pd.concat(
    [pod_column_profile, job_column_profile],
    ignore_index=True,
)[
    ["dataset", "column_name", "column_type", "min", "max",
     "count", "null_percentage"]
]
display(column_profile)


In [ ]:
def numeric_profile(table, dataset, columns):
    expressions = ["count(*) AS total_rows"]
    for column in columns:
        expressions.extend([
            f'count("{column}") AS "{column}__non_null"',
            f'min("{column}") AS "{column}__min"',
            f'quantile_cont("{column}", 0.25) AS "{column}__q25"',
            f'quantile_cont("{column}", 0.50) AS "{column}__median"',
            f'quantile_cont("{column}", 0.75) AS "{column}__q75"',
            f'quantile_cont("{column}", 0.99) AS "{column}__p99"',
            f'max("{column}") AS "{column}__max"',
            f'avg("{column}") AS "{column}__mean"',
            f'stddev_pop("{column}") AS "{column}__sd"',
        ])
    row = con.execute(
        f"SELECT {', '.join(expressions)} FROM {table}"
    ).df().iloc[0]
    total = int(row["total_rows"])
    records = []
    for column in columns:
        records.append({
            "dataset": dataset,
            "column": column,
            "non_null_count": int(row[f"{column}__non_null"]),
            "non_null_percent": 100 * float(row[f"{column}__non_null"]) / total,
            "min": row[f"{column}__min"],
            "q25": row[f"{column}__q25"],
            "median": row[f"{column}__median"],
            "q75": row[f"{column}__q75"],
            "p99": row[f"{column}__p99"],
            "max": row[f"{column}__max"],
            "mean": row[f"{column}__mean"],
            "sd": row[f"{column}__sd"],
        })
    return pd.DataFrame(records)

pod_numeric = numeric_profile("pod_profile_analysis", pod_scope_label, [
    "gpu_request", "used_gpu_hours", "avg_gpu_sm_util",
    "avg_gpu_mem_gib", "cpu_request_cores",
    "schedule_delay_sec_clean", "ready_delay_sec_clean",
])
job_numeric = numeric_profile("job_profile_analysis", job_scope_label, [
    "gpu_request", "duration_hours",
    "schedule_delay_sec_clean", "ready_delay_sec_clean",
])
numeric_distribution = pd.concat([pod_numeric, job_numeric], ignore_index=True)
numeric_columns = ["non_null_percent", "min", "q25", "median",
                   "q75", "p99", "max", "mean", "sd"]
numeric_distribution[numeric_columns] = (
    numeric_distribution[numeric_columns].astype(float).round(6)
)
display(numeric_distribution)


In [ ]:
def top_categories(table, dataset, columns, top_n=10):
    total = con.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
    frames = []
    for column in columns:
        frame = con.execute(f'''            SELECT
                '{dataset}' AS dataset,
                '{column}' AS column,
                coalesce(cast("{column}" AS VARCHAR), '<NULL>') AS value,
                count(*) AS count
            FROM {table}
            GROUP BY "{column}"
            ORDER BY count DESC
            LIMIT {int(top_n)}
        ''').df()
        frame["percent"] = 100 * frame["count"] / total
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

pod_categories = top_categories("pod_profile_analysis", pod_scope_label, [
    "state_public", "gpu_spec_public", "priority_class",
    "job_type_public", "model_type_public", "is_genai_request",
    "ready_status", "workload_id_status",
])
job_categories = top_categories("job_profile_analysis", job_scope_label, [
    "gpu_spec_public", "priority_class", "job_type_public",
    "model_type_public", "is_genai_request", "ready_status",
    "schedule_status", "workload_id_status",
])
categorical_distribution = pd.concat(
    [pod_categories, job_categories],
    ignore_index=True,
)
display(categorical_distribution)


In [ ]:
# Export small, shareable profile tables (not the raw datasets).
PROFILE_DIR = ROOT / "reports" / "data_profile"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

profile_tables = {
    "file_inventory.csv": file_inventory,
    "quality_report.csv": quality_report,
    "workload_id_coverage.csv": workload_id_coverage,
    "column_profile.csv": column_profile,
    "numeric_distribution.csv": numeric_distribution,
    "categorical_distribution.csv": categorical_distribution,
}

export_manifest = []
for filename, table in profile_tables.items():
    output_path = PROFILE_DIR / filename
    table.to_csv(output_path, index=False)
    export_manifest.append({
        "file": output_path.relative_to(ROOT).as_posix(),
        "rows": len(table),
        "columns": len(table.columns),
    })

export_manifest = pd.DataFrame(export_manifest)
display(export_manifest)
print("Profile tables exported to:", PROFILE_DIR)


## 組員使用與後續分析

- 完整度檢查涵蓋兩個完整 Parquet；profile 表格使用明確標示的快速範圍。
- 六份小型 profile CSV 會輸出至 `reports/data_profile/`。
- 組員只需要 GitHub repo、共享 ZIP，以及 `r_start.R`，不需要 720 個 hourly files。
- 原始 Parquet 永遠保持不變；補值、刪列與欄位轉換在查詢時完成。
- 兩份資料若要 join，應先各自彙總到明確層級並報告 match coverage，不可直接 raw join。
